# VANGROVE - Data Wrangling & Cleaning Pipeline 🌱

**Project:** VANGROVE (Visual Analytics & Navigation for Geographic Regional Output & Variety Evaluation)

## Deskripsi Notebook
Notebook ini berfokus pada tahap awal persiapan data (*Data Preparation*), yang meliputi:
1. **Data Gathering & Standardization:** Menggabungkan dataset daun tanaman (Jagung, Tomat, Mangga, Kentang) dari berbagai sumber dan menstandardisasi penamaan kelas penyakit.
2. **Data Assessing & Cleaning:** Mendeteksi dan menghapus duplikasi gambar menggunakan *MD5 hashing*, serta menghapus file gambar yang *corrupt*.
3. **Synthetic Metadata Engineering:** Menambahkan data sintetis berupa tanggal dan koordinat lokasi geografis (`lat`, `lon`) untuk mendukung fitur Web-GIS pada *dashboard*.
4. **Exploratory Data Analysis (EDA):** Visualisasi distribusi akhir kelas penyakit sebelum masuk ke tahap *modeling*.

**Output:** `cleaned_dataset.csv` yang siap digunakan untuk *train-test split* dan pelatihan model CNN.

In [162]:
import os
import shutil
import hashlib
from PIL import Image
import pandas as pd
from collections import Counter
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
IN_COLAB = 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive #type: ignore
    drive.mount('/content/drive')

## Data Gathering
Dataset dikumpulkan dari beberapa sumber dan digabungkan ke dalam satu struktur folder berdasarkan jenis tanaman dan penyakit

In [3]:
if IN_COLAB:
    print("Running on Google Colab")
    BASE_PATH = "/content/drive/MyDrive/VANGROVE/data"
else:
    print("Running on Local")
    BASE_PATH = "data"

RAW_PATH = os.path.join(BASE_PATH, "raw")
COMBINED_PATH = os.path.join(BASE_PATH, "combined")

os.makedirs(COMBINED_PATH, exist_ok=True)

Running on Local


In [4]:
dataset_mapping = {
    "corn-leaf-disease": "corn",
    "corn-or-maize-leaf-disease-dataset": "corn",
    "mango-leaf-disease-dataset": "mango",
    "potato-leaf-disease-dataset": "potato",
    "tomato-disease-multiple-sources": "tomato",
    "tomato-leaves-dataset": "tomato"
}

Dataset dari berbagai sumber dipetakan ke dalam kategori tanaman: corn; mango; potato; tomato

In [5]:
def clean_name(name):
    return name.lower().replace(" ", "_").replace("-", "_")

In [6]:
corn_mapping = {
    "daun_sehat": "healthy",
    "healthy": "healthy",

    "karat_daun": "rust",
    "common_rust": "rust",

    "hawar_daun": "blight",
    "blight": "blight",

    "bercak_daun": "leaf_spot",
    "gray_leaf_spot": "leaf_spot"
}

In [7]:
def short_name(filename):
    name, ext = os.path.splitext(filename)
    return hashlib.md5(name.encode()).hexdigest() + ext.lower()

Dilakukan standardisasi nama folder dan file untuk menghindari inkonsistensi data

In [8]:
def process_folder(source_path, plant_name):
    file_count = 0
    error_count = 0

    mapping = corn_mapping if plant_name == "corn" else {}

    for disease in os.listdir(source_path):
        disease_path = os.path.join(source_path, disease)

        if not os.path.isdir(disease_path):
            continue

        clean_disease = clean_name(disease)
        final_disease = mapping.get(clean_disease, clean_disease)

        dest_folder = os.path.join(COMBINED_PATH, plant_name, final_disease)
        os.makedirs(dest_folder, exist_ok=True)

        for file in os.listdir(disease_path):
            src_file = os.path.join(disease_path, file)

            if not os.path.isfile(src_file):
                continue

            try:
                new_name = short_name(file)
                dst_file = os.path.join(dest_folder, new_name)

                # copy file
                if not os.path.exists(dst_file):
                    shutil.copy(src_file, dst_file)
                    file_count += 1

            except Exception as e:
                error_count += 1
                print(f"skip: {file}")

    return file_count, error_count

Menggabungkan seluruh dataset ke dalam folder combined dengan struktur: plant -> disease -> images

In [9]:
# reset
shutil.rmtree(COMBINED_PATH, ignore_errors=True)
os.makedirs(COMBINED_PATH, exist_ok=True)

# melanjutkan run
for dataset_name, plant_name in dataset_mapping.items():
    dataset_path = os.path.join(RAW_PATH, dataset_name)

    if not os.path.exists(dataset_path):
        continue

    subfolders = os.listdir(dataset_path)

    if "train" in subfolders or "valid" in subfolders:
        for split in ["train", "valid"]:
            split_path = os.path.join(dataset_path, split)
            if os.path.exists(split_path):
                files, errors = process_folder(split_path, plant_name)
    else:
        files, errors = process_folder(dataset_path, plant_name)

Folder combined selalu di-reset setiap kali proses dijalankan ulang. Hal ini dilakukan agar tahap assessing dilakukan pada data yang belum dibersihkan (masih mengandung duplikat dan data rusak)

## Data Assessing
Dilakukan evaluasi terhadap kualitas dataset, meliputi:
- Deteksi data duplikat
- Identifikasi gambar rusak
- Pemeriksaan format file

In [10]:
file_paths = []

for plant in os.listdir(COMBINED_PATH):
    plant_path = os.path.join(COMBINED_PATH, plant)

    for disease in os.listdir(plant_path):
        disease_path = os.path.join(plant_path, disease)

        for img in os.listdir(disease_path):
            img_path = os.path.join(disease_path, img)

            file_paths.append((plant, disease, img_path))

In [11]:
def get_image_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = {}
duplicates = []

for plant, disease, path in file_paths:
    try:
        img_hash = get_image_hash(path)

        if img_hash in hashes:
            duplicates.append(path)
        else:
            hashes[img_hash] = path

    except:
        pass

print("Jumlah duplikat:", len(duplicates))

Jumlah duplikat: 1166


In [12]:
bad_images = []

for _, _, path in file_paths:
    try:
        Image.open(path).verify()
    except:
        bad_images.append(path)

print("Gambar rusak:", len(bad_images))

Gambar rusak: 1


In [13]:
exts = set()

for _, _, path in file_paths:
    exts.add(path.split('.')[-1])

print(exts)

{'jpg', 'jpeg', 'png'}


### Insight
- Ditemukan data duplikat yang kemungkinan berasal dari penggabungan dataset berbeda
- Terdapat beberapa gambar yang tidak valid (corrupt)
- Format file bervariasi (jpg, jpeg, png)

## Data Cleaning
Dilakukan pembersihan dataset dengan:
- Menghapus data duplikat
- Menghapus gambar rusak
- Analisis distribusi data

In [14]:
for path in duplicates:
    os.remove(path)

print("Duplikat terhapus:", len(duplicates))

Duplikat terhapus: 1166


In [15]:
deleted = 0

for path in bad_images:
    if os.path.exists(path):
        os.remove(path)
        deleted += 1

print("Gambar rusak terhapus:", deleted)

Gambar rusak terhapus: 1


In [16]:
file_paths = []

for plant in os.listdir(COMBINED_PATH):
    for disease in os.listdir(os.path.join(COMBINED_PATH, plant)):
        for img in os.listdir(os.path.join(COMBINED_PATH, plant, disease)):
            file_paths.append((plant, disease, os.path.join(COMBINED_PATH, plant, disease, img)))

In [17]:
print("Total gambar:", len(file_paths))

Total gambar: 46055


In [18]:
counter = Counter()

for plant, disease, path in file_paths:
    counter[(plant, disease)] += 1

df_count = pd.DataFrame(
    [(p, d, c) for (p, d), c in counter.items()],
    columns=["plant", "disease", "count"]
)

df_count

,plant,disease,count
0,corn,blight,2146
1,corn,healthy,2162
2,corn,leaf_spot,998
3,corn,rust,2300
4,mango,anthracnose,486
5,mango,bacterial_canker,500
6,mango,cutting_weevil,500
7,mango,die_back,493
8,mango,gall_midge,500
9,mango,healthy,500


Insight
- Data duplikat berhasil dihapus sehingga mengurangi potensi bias
- Gambar corrupt dihapus untuk menjaga kualitas dataset
- Distribusi data belum sepenuhnya seimbang, terutama pada dataset potato

## Feature Engineering

Pada tahap ini dilakukan penambahan fitur baru untuk memperkaya informasi pada dataset. Fitur yang ditambahkan meliputi kondisi lingkungan (condition), lokasi, waktu (date), serta rekomendasi penanganan (recommendation) untuk setiap jenis penyakit tanaman.


In [61]:
data = []

# 1. HAPUS TANDA KUTIP (Gunakan variabel yang sudah ada)
base_path = COMBINED_PATH

locations = ["Sumatera", "Jawa", "Kalimantan", "Sulawesi"]

# 2. Pindahkan kamus koordinat ke sini
location_map = {
    "Sumatera": (2.9507563, 98.7846727), #Simalungun
    "Jawa": (-7.331565, 109.397446), #Purbalingga
    "Kalimantan": (0.370236, 109.922358), #Ngabang
    "Sulawesi": (-3.449786, 119.776606) #Enrekang
}

for plant, disease, path in file_paths:
    # RANDOM tanggal (misalnya 2023-2025)
    start = datetime(2023, 1, 1)
    end = datetime(2024, 11, 30)

    delta_days = (end - start).days
    random_date = start + timedelta(days=random.randint(0, delta_days))

    # Pilih lokasi secara acak dan simpan di variabel
    chosen_location = random.choice(locations)

    # Masukkan semua data sekaligus
    data.append({
        "plant": plant,
        "disease": disease,
        "image_path": path,
        "date": random_date,
        "location": chosen_location,
        "lat": location_map[chosen_location][0], # Langsung ambil Latitude
        "lon": location_map[chosen_location][1]  # Langsung ambil Longitude
    })

df_cleaned = pd.DataFrame(data)

Data waktu dan lokasi ditambahkan secara simulatif untuk mendukung analisis eksploratif dan visualisasi dashboard.

In [62]:
disease_info = {

    # ================= CORN =================
    "blight": {
        "plant": "corn",
        "condition": ["kelembaban tinggi", "nitrogen berlebih", "sanitasi buruk"],
        "recommendation": [
            "Kurangi penggunaan pupuk nitrogen berlebih",
            "Perbaiki sanitasi lahan",
            "Gunakan benih sehat"
        ]
    },

    "healthy": {
        "plant": "all",
        "condition": ["optimal"],
        "recommendation": [
            "Pertahankan kondisi tanaman",
            "Lakukan perawatan rutin"
        ]
    },

    "leaf_spot": {
        "plant": "corn",
        "condition": ["kelembaban tinggi", "tanah kurang subur", "residu tanaman"],
        "recommendation": [
            "Gunakan pupuk yang cukup",
            "Bersihkan sisa tanaman",
            "Perbaiki kualitas tanah"
        ]
    },

    "rust": {
        "plant": "corn",
        "condition": ["kelembaban tinggi", "lingkungan lembap"],
        "recommendation": [
            "Kurangi kelembaban berlebih",
            "Gunakan varietas tahan",
            "Lakukan monitoring rutin"
        ]
    },

    # ================= MANGO =================
    "anthracnose": {
        "plant": "mango",
        "condition": ["lembap", "musim hujan"],
        "recommendation": [
            "Pangkas bagian terinfeksi",
            "Gunakan varietas tahan",
            "Jaga kebersihan kebun"
        ]
    },

    "bacterial_canker": {
        "plant": "mango",
        "condition": ["infeksi bakteri"],
        "recommendation": [
            "Pangkas dan bakar bagian terinfeksi",
            "Lakukan sanitasi kebun"
        ]
    },

    "cutting_weevil": {
        "plant": "mango",
        "condition": ["serangan hama", "batang/ranting rusak"],
        "recommendation": [
            "Pangkas dan bakar cabang terinfeksi",
            "Gunakan agen hayati",
            "Gunakan perangkap feromon"
        ]
    },

    "die_back": {
        "plant": "mango",
        "condition": ["infeksi pada cabang", "lingkungan kurang sehat"],
        "recommendation": [
            "Pangkas bagian yang mati",
            "Jaga kesehatan tanaman",
            "Perbaiki perawatan tanaman"
        ]
    },

    "gall_midge": {
        "plant": "mango",
        "condition": ["kelembaban tinggi", "suhu sedang"],
        "recommendation": [
            "Pantau tanaman secara rutin",
            "Gunakan perangkap serangga",
            "Bersihkan area kebun"
        ]
    },

    "healthy": {
        "plant": "mango",
        "condition": ["optimal"],
        "recommendation": [
            "Pertahankan kondisi tanaman",
            "Lakukan perawatan rutin"
        ]
    },

    "powdery_mildew": {
        "plant": "mango",
        "condition": ["kelembaban tinggi", "ventilasi buruk"],
        "recommendation": [
            "Tingkatkan sirkulasi udara",
            "Pangkas daun",
            "Gunakan fungisida alami"
        ]
    },

    "sooty_mould": {
        "plant": "mango",
        "condition": ["hama penghisap"],
        "recommendation": [
            "Kendalikan hama",
            "Bersihkan daun",
            "Gunakan pestisida alami"
        ]
    },

    # ================= POTATO =================
    "bacteria": {
        "plant": "potato",
        "condition": ["infeksi bakteri"],
        "recommendation": [
            "Gunakan benih sehat",
            "Buang tanaman terinfeksi",
            "Jaga kebersihan alat"
        ]
    },

    "fungi": {
        "plant": "potato",
        "condition": ["dingin", "kelembaban tinggi"],
        "recommendation": [
            "Perbaiki drainase",
            "Gunakan fungisida",
            "Atur jarak tanam"
        ]
    },

    "healthy": {
        "plant": "potato",
        "condition": ["optimal"],
        "recommendation": [
            "Pertahankan kondisi tanaman",
            "Lakukan perawatan rutin"
        ]
    },

    "nematode": {
        "plant": "potato",
        "condition": ["tanah terinfeksi", "lingkungan lembap"],
        "recommendation": [
            "Gunakan rotasi tanaman",
            "Perbaiki kualitas tanah",
            "Gunakan varietas tahan"
        ]
    },

    "pest": {
        "plant": "potato",
        "condition": ["musim kemarau"],
        "recommendation": [
            "Lakukan monitoring rutin",
            "Gunakan pestisida alami",
            "Cek daun secara berkala"
        ]
    },

    "phytopthora": {
        "plant": "potato",
        "condition": ["dingin", "kelembaban tinggi", "musim hujan"],
        "recommendation": [
            "Gunakan fungisida",
            "Perbaiki drainase",
            "Atur jarak tanam"
        ]
    },

    "virus": {
        "plant": "potato",
        "condition": ["penularan serangga"],
        "recommendation": [
            "Gunakan benih bersertifikat",
            "Cabut tanaman terinfeksi",
            "Bersihkan alat"
        ]
    },

    # ================= TOMATO =================
    "bacterial_spot": {
        "plant": "tomato",
        "condition": ["lembap", "air mengenai daun"],
        "recommendation": [
            "Hindari penyiraman ke daun",
            "Gunakan benih sehat",
            "Bersihkan lahan"
        ]
    },

    "early_blight": {
        "plant": "tomato",
        "condition": ["kelembaban tinggi", "tanah terinfeksi"],
        "recommendation": [
            "Buang daun terinfeksi",
            "Jaga jarak tanam",
            "Gunakan fungisida"
        ]
    },

    "healthy": {
        "plant": "tomato",
        "condition": ["optimal"],
        "recommendation": [
            "Pertahankan kondisi tanaman",
            "Lakukan perawatan rutin"
        ]
    },

    "late_blight": {
        "plant": "tomato",
        "condition": ["hujan", "kelembaban tinggi"],
        "recommendation": [
            "Perbaiki sirkulasi udara",
            "Gunakan fungisida",
            "Kurangi kelembaban"
        ]
    },

    "leaf_mold": {
        "plant": "tomato",
        "condition": ["kelembaban >85%", "ventilasi buruk"],
        "recommendation": [
            "Perbaiki ventilasi",
            "Kurangi kelembaban",
            "Pangkas daun"
        ]
    },

    "powdery_mildew": {
        "plant": "tomato",
        "condition": ["kering + lembap", "cahaya rendah"],
        "recommendation": [
            "Tingkatkan cahaya",
            "Gunakan fungisida",
            "Buang daun terinfeksi"
        ]
    },

    "septoria_leaf_spot": {
        "plant": "tomato",
        "condition": ["kelembaban tinggi", "percikan air"],
        "recommendation": [
            "Hindari penyiraman ke daun",
            "Buang daun terinfeksi",
            "Tingkatkan sirkulasi udara"
        ]
    },

    "spider_mites_two_spotted_spider_mite": {
        "plant": "tomato",
        "condition": ["panas", "kering"],
        "recommendation": [
            "Semprot air ke daun",
            "Gunakan neem oil",
            "Bersihkan gulma"
        ]
    },

    "target_spot": {
        "plant": "tomato",
        "condition": ["kelembaban tinggi", "cuaca ekstrem"],
        "recommendation": [
            "Lakukan monitoring rutin",
            "Gunakan fungisida",
            "Perbaiki kondisi lingkungan"
        ]
    },

    "tomato_mosaic_virus": {
        "plant": "tomato",
        "condition": ["virus", "alat terkontaminasi"],
        "recommendation": [
            "Cabut tanaman terinfeksi",
            "Bersihkan alat",
            "Gunakan benih sehat"
        ]
    },

    "tomato_yellow_leaf_curl_virus": {
        "plant": "tomato",
        "condition": ["serangga", "cuaca panas"],
        "recommendation": [
            "Gunakan jaring pelindung",
            "Kontrol lalat putih",
            "Gunakan varietas tahan"
        ]
    }
}

Penambahan kondisi dilakukan berdasarkan faktor-faktor yang memengaruhi munculnya penyakit, seperti kelembaban, suhu, dan kondisi lingkungan lainnya. Sementara itu, rekomendasi disusun sebagai bentuk tindakan yang dapat dilakukan untuk mencegah atau menangani penyakit tersebut

In [63]:
def get_info(disease, key):
    return disease_info.get(disease, {}).get(key, [])

df_cleaned["condition"] = df_cleaned["disease"].apply(lambda x: get_info(x, "condition"))
df_cleaned["recommendation"] = df_cleaned["disease"].apply(lambda x: get_info(x, "recommendation"))

In [64]:
df_cleaned.info()
df_cleaned.describe(include='all')
df_cleaned.head()

<class 'pandas.DataFrame'>
RangeIndex: 46055 entries, 0 to 46054
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   plant           46055 non-null  str           
 1   disease         46055 non-null  str           
 2   image_path      46055 non-null  str           
 3   date            46055 non-null  datetime64[us]
 4   location        46055 non-null  str           
 5   lat             46055 non-null  float64       
 6   lon             46055 non-null  float64       
 7   condition       46055 non-null  object        
 8   recommendation  46055 non-null  object        
dtypes: datetime64[us](1), float64(2), object(2), str(4)
memory usage: 7.4+ MB


,plant,disease,image_path,date,location,lat,lon,condition,recommendation
0,corn,blight,data\combined\corn\blight\00008317a7eac191b0a3...,2024-08-12,Sulawesi,-3.449786,119.776606,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P..."
1,corn,blight,data\combined\corn\blight\001e67fe3fa91f6b333b...,2024-07-03,Kalimantan,0.370236,109.922358,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P..."
2,corn,blight,data\combined\corn\blight\006b1179d992f97c2a4b...,2024-04-22,Sumatera,2.950756,98.784673,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P..."
3,corn,blight,data\combined\corn\blight\0071370b19d8fbb1554b...,2023-12-20,Jawa,-7.331565,109.397446,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P..."
4,corn,blight,data\combined\corn\blight\008870ea42867952619c...,2023-12-28,Sulawesi,-3.449786,119.776606,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P..."


## Exploratory Data Analysis

In [65]:
df_cleaned["disease"].value_counts()

disease
healthy                                 6568
late_blight                             3732
septoria_leaf_spot                      3621
bacterial_spot                          3558
leaf_mold                               3094
early_blight                            2746
tomato_mosaic_virus                     2736
tomato_yellow_leaf_curl_virus           2534
rust                                    2300
target_spot                             2284
spider_mites_two_spotted_spider_mite    2182
blight                                  2146
powdery_mildew                          1756
leaf_spot                                998
fungi                                    743
pest                                     602
bacteria                                 569
virus                                    530
bacterial_canker                         500
cutting_weevil                           500
gall_midge                               500
sooty_mould                              500
di

In [66]:
df_cleaned["plant"].value_counts()

plant
tomato    31449
corn       7606
mango      3979
potato     3021
Name: count, dtype: int64

In [67]:
pd.crosstab(df_cleaned["plant"], df_cleaned["disease"]).T

plant,corn,mango,potato,tomato
disease,,,,
anthracnose,0,486,0,0
bacteria,0,0,569,0
bacterial_canker,0,500,0,0
bacterial_spot,0,0,0,3558
blight,2146,0,0,0
cutting_weevil,0,500,0,0
die_back,0,493,0,0
early_blight,0,0,0,2746
fungi,0,0,743,0


In [68]:
pd.crosstab(df_cleaned["location"], df_cleaned["plant"])

plant,corn,mango,potato,tomato
location,,,,
Jawa,1874,1008,766,7778
Kalimantan,1926,1017,767,7865
Sulawesi,1904,979,731,7918
Sumatera,1902,975,757,7888


In [69]:
print("\n--- Rentang Waktu Dataset ---")
print("Dari:", df_cleaned['date'].min())
print("Sampai:", df_cleaned['date'].max())


--- Rentang Waktu Dataset ---
Dari: 2023-01-01 00:00:00
Sampai: 2024-11-30 00:00:00


In [70]:
df_cleaned["year"] = df_cleaned["date"].dt.year
df_cleaned["month"] = df_cleaned["date"].dt.month

df_cleaned.groupby(["year", "month"]).size()

year  month
2023  1        2021
      2        1854
      3        2000
      4        1992
      5        2021
      6        1928
      7        2078
      8        2101
      9        1987
      10       2057
      11       2007
      12       1992
2024  1        2030
      2        1944
      3        2072
      4        1903
      5        2136
      6        1974
      7        2032
      8        2061
      9        1898
      10       2004
      11       1963
dtype: int64

In [71]:
df_cleaned.explode("condition")["condition"].value_counts()

condition
kelembaban tinggi          19379
optimal                     6568
lembap                      4044
hujan                       3732
percikan air                3621
air mengenai daun           3558
kelembaban >85%             3094
ventilasi buruk             3094
tanah terinfeksi            2814
virus                       2736
alat terkontaminasi         2736
serangga                    2534
cuaca panas                 2534
lingkungan lembap           2368
cuaca ekstrem               2284
panas                       2182
kering                      2182
nitrogen berlebih           2146
sanitasi buruk              2146
kering + lembap             1756
cahaya rendah               1756
infeksi bakteri             1069
dingin                      1052
tanah kurang subur           998
residu tanaman               998
musim hujan                  795
musim kemarau                602
penularan serangga           530
serangan hama                500
batang/ranting rusak         500


In [72]:
df_exploded = df_cleaned.explode("condition").reset_index(drop=True)

pd.crosstab(
    df_exploded["condition"],
    df_exploded["disease"]
)

disease,anthracnose,bacteria,bacterial_canker,bacterial_spot,blight,cutting_weevil,die_back,early_blight,fungi,gall_midge,...,phytopthora,powdery_mildew,rust,septoria_leaf_spot,sooty_mould,spider_mites_two_spotted_spider_mite,target_spot,tomato_mosaic_virus,tomato_yellow_leaf_curl_virus,virus
condition,,,,,,,,,,,,,,,,,,,,,
air mengenai daun,0,0,0,3558,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
alat terkontaminasi,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2736,0,0
batang/ranting rusak,0,0,0,0,0,500,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
cahaya rendah,0,0,0,0,0,0,0,0,0,0,...,0,1756,0,0,0,0,0,0,0,0
cuaca ekstrem,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,2284,0,0,0
cuaca panas,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2534,0
dingin,0,0,0,0,0,0,0,0,743,0,...,309,0,0,0,0,0,0,0,0,0
hama penghisap,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,500,0,0,0,0,0
hujan,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Insight

1. Dataset didominasi oleh tanaman tomat dengan jumlah data yang jauh lebih besar dibandingkan tanaman lain seperti jagung, mangga, dan kentang. Hal ini menunjukkan bahwa analisis pada tanaman tomat akan lebih detail dan representatif dibandingkan tanaman lainnya
2. Dataset terdiri dari dua kategori utama, yaitu kondisi sehat (healthy) dan berbagai jenis penyakit. Kelas healthy memiliki jumlah data terbanyak, yang menunjukkan bahwa dataset juga mencakup banyak contoh tanaman dalam kondisi normal. Jika hanya mempertimbangkan kelas penyakit, maka penyakit yang paling dominan adalah late_blight (3732), diikuti oleh septoria_leaf_spot (3621), dan bacterial_spot (3558)
3. Setiap jenis penyakit hanya muncul pada tanaman tertentu tanpa adanya overlap antar tanaman. Hal ini menunjukkan bahwa proses penggabungan dan pelabelan data telah dilakukan dengan baik dan konsisten
4. Distribusi data antar wilayah relatif merata, yang menunjukkan bahwa data lokasi bersifat simulatif dan tidak merepresentasikan kondisi geografis nyata, melainkan digunakan untuk keperluan analisis eksploratif
5. Distribusi jumlah data per bulan cenderung merata sepanjang periode 2023–2024, yang menunjukkan bahwa data waktu dihasilkan secara acak dan tidak mencerminkan pola musiman tertentu
6. Kondisi lingkungan yang paling sering muncul adalah kelembaban tinggi, yang menunjukkan bahwa faktor ini sering dikaitkan dengan kemunculan berbagai jenis penyakit dalam dataset
7. Terdapat hubungan yang jelas antara kondisi lingkungan dan jenis penyakit. Misalnya, kelembaban tinggi berkaitan dengan berbagai penyakit, sementara kondisi spesifik seperti alat terkontaminasi dan serangga berkaitan dengan penyakit berbasis virus. Hal ini menunjukkan bahwa pendekatan berbasis kondisi dapat digunakan untuk memberikan rekomendasi penanganan
8. Terdapat ketidakseimbangan jumlah data antar kelas penyakit, di mana beberapa kelas memiliki jumlah data yang sangat kecil dibandingkan yang lain. Hal ini berpotensi memengaruhi performa model dalam tahap klasifikasi

## Visualization & Explanatory Analysis

Pertanyaan 1: Tanaman apa yang memiliki jumlah kasus penyakit terbanyak setiap bulan selama periode Januari 2023 hingga November 2024?

In [73]:
df_disease = df_cleaned[df_cleaned["disease"] != "healthy"]

In [166]:
df_disease["year_month"] = df_disease["date"].dt.to_period("M")
result = df_disease.groupby(["year_month", "plant"]).size().unstack()
df_disease.head()

,plant,disease,image_path,date,location,lat,lon,condition,recommendation,year,month,year_month
0,corn,blight,data\combined\corn\blight\00008317a7eac191b0a3...,2024-08-12,Sulawesi,-3.449786,119.776606,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P...",2024,8,2024-08
1,corn,blight,data\combined\corn\blight\001e67fe3fa91f6b333b...,2024-07-03,Kalimantan,0.370236,109.922358,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P...",2024,7,2024-07
2,corn,blight,data\combined\corn\blight\006b1179d992f97c2a4b...,2024-04-22,Sumatera,2.950756,98.784673,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P...",2024,4,2024-04
3,corn,blight,data\combined\corn\blight\0071370b19d8fbb1554b...,2023-12-20,Jawa,-7.331565,109.397446,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P...",2023,12,2023-12
4,corn,blight,data\combined\corn\blight\008870ea42867952619c...,2023-12-28,Sulawesi,-3.449786,119.776606,"[kelembaban tinggi, nitrogen berlebih, sanitas...","[Kurangi penggunaan pupuk nitrogen berlebih, P...",2023,12,2023-12


Pertanyaan 2: Apa jenis penyakit yang paling dominan pada setiap tanaman (tomat, jagung, mangga, dan kentang) berdasarkan jumlah kasus dalam dataset?

Pertanyaan 3: Bagaimana distribusi jumlah kasus penyakit tanaman pada setiap wilayah (Sumatera, Jawa, Kalimantan, dan Sulawesi) selama periode pengamatan?

Pertanyaan 4: Kondisi lingkungan apa yang paling sering terkait dengan kemunculan masing-masing jenis penyakit tanaman?

Pertanyaan 5: Apakah terdapat tren peningkatan atau penurunan jumlah kasus penyakit tanaman dari bulan ke bulan selama periode Januari 2023 hingga November 2024?